In [ ]:
import pandas as pd
import scanpy as sc
import scipy as sp
import anndata as ad
import infercnvpy as cnv
import rpy2
import numpy as np
from scipy.sparse import issparse

# InferCNV

In [ ]:
# Load data
adata = sc.read_h5ad('../data/adata/GBM_LEAP_15_05_23_coarse_annos.h5ad')
gtf = "../data/metadata/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/genes/genes.gtf"

# Per-patient CNV calling

In [ ]:
for donor in sorted(set(adata.obs['donor_id'])):
    print(donor)
    ## RUN INFERCNV
    # separate per donor
    ad_donor = adata[adata.obs['donor_id']==donor,:].copy()
    ad_donor.layers['counts'] = ad_donor.X.copy()
    if 'log1p' in ad_donor.uns.keys():
        del(ad_donor.uns['log1p'])

    # remove already annotated columns
    remove = [i for i in [str(i) for i in range(1,23)] if i in ad_donor.obs.columns]
    ad_donor.obs = ad_donor.obs.drop(remove, axis=1)

    # log transform and normalise data
    sc.pp.normalize_total(ad_donor, target_sum=1e4)
    sc.pp.log1p(ad_donor)

    # prep adata with gtf positions
    cnv.io.genomic_position_from_gtf(gtf, ad_donor)

    # call CNVs
    cnv.tl.infercnv(
        ad_donor,
        reference_key='coarse_prediction',
        reference_cat=[
            'Macrophages', 'Oligodendrocytes', 
            'Neurons (Exc)', 'Neurons (Inh)', 'Lymphocytes', 'Vascular-associated'],
        window_size=100,
        lfc_clip = 3,
        step=1
    )


    ## PLOT INFERCNV RESULTS
    use_rep = 'cnv'
    tmp_ad_donor = ad.AnnData(X=ad_donor.obsm[f"X_{use_rep}"], obs=ad_donor.obs)
    groupby = 'leiden_scVI'
    figsize = (16, 10)
    cmap = 'bwr'

    # re-sort, as saving & loading anndata destroys the order
    chr_pos_dict = dict(
            sorted(ad_donor.uns[use_rep]["chr_pos"].items(), key=lambda x: x[1])
        )
    chr_pos = list(chr_pos_dict.values())

    # center color map at 0
    tmp_data = tmp_ad_donor.X.data if issparse(tmp_ad_donor.X) else tmp_ad_donor.X


    # remove already annotated columns
    remove = [i for i in [str(i) for i in range(1,23)] if i in tmp_ad_donor.obs.columns]
    tmp_ad_donor.obs = tmp_ad_donor.obs.drop(remove, axis=1)

    # add chromosome annotations
    var_group_positions = list(zip(chr_pos, chr_pos[1:] + [tmp_ad_donor.shape[1]]))

    # plot heatmap
    return_ax_dic = sc.pl.heatmap(
            tmp_ad_donor,
            var_names=tmp_ad_donor.var.index,
            groupby=groupby,
            figsize=figsize,
            cmap=cmap,
            show_gene_labels=False,
            var_group_positions=var_group_positions,
            var_group_labels=list(chr_pos_dict.keys()),
            vmin=-0.2, vcenter=0, vmax=0.2,
            show=False, dendrogram=False,
            save = '_{}_CNV_heatmap_infercnvpy.png'.format(donor)
        )

    ## CLUSTER INFERCNV
    cnv.tl.pca(ad_donor)
    cnv.pp.neighbors(ad_donor)
    cnv.tl.leiden(ad_donor)
    cnv.tl.umap(ad_donor)
    cnv.tl.cnv_score(ad_donor)

    ## ADDITIONAL PLOTS
    import matplotlib.pyplot as plt
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(11, 11))
    ax4.axis("off")
    cnv.pl.umap(
        ad_donor,
        color="cnv_leiden",
        legend_loc="on data",
        legend_fontoutline=2,
        ax=ax1,
        show=False,
    )
    cnv.pl.umap(ad_donor, color="cnv_score", ax=ax2, show=False)
    cnv.pl.umap(ad_donor, color="coarse_prediction", ax=ax3)


    ## CALCULATE CNV SCORES
    # create dataframe for easy merging with cluster info
    chromosome_positions = []
    chromosome = 0
    for i in var_group_positions:
        chromosome += 1
        chr_range = list(range(i[0], i[1]))
        chrdf = pd.DataFrame({'chromosome':[chromosome]*len(chr_range),
                              'index':chr_range})
        chromosome_positions.append(chrdf)
    chromosome_positions = pd.concat(chromosome_positions, 
                                     ignore_index=True)
    # splice array to pull out just chr7 and 10
    indices = chromosome_positions[(chromosome_positions['chromosome']==7) |
                         (chromosome_positions['chromosome']==10)]['index'].tolist()

    diagnostic_chrs_array = ad_donor.obsm['X_cnv'][:,indices]

    # Squared sum
    nrow,ncol = diagnostic_chrs_array.shape
    squared_sum = diagnostic_chrs_array.multiply(diagnostic_chrs_array).sum(1)/ncol
    ad_donor.obs['CNV_signal_sqsum'] = squared_sum

    # absolute mean
    ad_donor.obs['CNV_signal_mean'] = np.absolute(diagnostic_chrs_array).mean(axis=1)
    cnv.pl.umap(ad_donor, color="CNV_signal_mean", vmax=0.02, 
                save = '_{}_CNV_signal_mean.png'.format(donor))
    
    ## CALCULATE CNV CORRELATION:
    # filter by non-annotated clusters and create a mean CNV signature from these
    non_tme = ad_donor[ad_donor.obs['coarse_prediction']=='Unknown',:]

    # get 
    x = ad_donor.obsm['X_cnv'].transpose()
    y = non_tme.obsm['X_cnv'].mean(axis=0).tolist()
    y = [item for sublist in y for item in sublist]
    y = np.array(y)

    # dimensions
    nrow,ncol = x.get_shape()

    # Correlation
    yy = y - y.mean()
    xm = x.mean(axis=0).A.ravel()
    ys = yy / np.sqrt(np.dot(yy, yy))
    xs = np.sqrt(np.add.reduceat(x.data**2, x.indptr[:-1]) - nrow*xm*xm)

    corr = np.add.reduceat(x.data * ys[x.indices], x.indptr[:-1]) / xs

    ad_donor.obs['cnv_corr'] = corr

    ## CNV export
    ad_donor.obs[['cell_id', 'leiden_scVI', 'coarse_prediction', 'cnv_leiden', 
                  'cnv_score', 'CNV_signal_sqsum', 'CNV_signal_mean', 'cnv_corr']]\
    .to_csv('../data/RNA_CNA/{}_CNA_info.csv'.format(donor), index = None)